# Load Data

In [92]:
# Load modular data 

import gurobipy as gp
from gurobipy import GRB
import pandas as pd

m = gp.Model("MIP")

from ranDataGen.order_data_loader import load_order_data

# Load from ranDataGen copy; change size and sample to switch datasets
size = "small"    # e.g. "small", "medium", "large"
sample = 1        # e.g. 1, 2, 3, ...

base_dir = f"ranDataGen copy/{size} sized samples/"

itemtypes_path = f"order_itemtypes_{sample}.csv"
quantities_path = f"order_quantities_{sample}.csv"
totes_path = f"orders_totes_{sample}.csv"

# Load baseline data using modular loader
df = load_order_data(
    itemtypes_path=itemtypes_path,
    quantities_path=quantities_path,
    totes_path=totes_path,
    base_dir=base_dir,
)

df

,order,item_slot,itemtype,quantity,tote
0,1,0,5,2,2
1,2,0,2,1,5
2,2,1,3,3,6
3,2,2,4,1,5
4,3,0,5,2,3
5,4,0,0,3,3
6,4,1,5,1,6
7,5,0,1,1,6
8,5,1,2,1,1
9,6,0,1,2,6


# Time Based Model

In [86]:
# ==================================================
# Build inventory items: (item_id, item_type, tote)
# ==================================================

items = []
tote_items = {}

item_id = 0

for _, row in df.iterrows():
    i = row["itemtype"]
    t = row["tote"]
    qty = int(row["quantity"])
    
    for q in range(qty):
        item_id += 1
        item = (item_id, i, t)
        items.append(item)
        
        if t not in tote_items:
            tote_items[t] = []
        tote_items[t].append(item)

N = len(items)
positions = range(1, N+1)

totes = list(tote_items.keys())

# number of items per tote
tote_size = {t: len(tote_items[t]) for t in totes}

# ==================================================
# Build demand dictionary
# ==================================================

orders = df["order"].unique()

demand = {}

for _, row in df.iterrows():
    o = row["order"]
    i = row["itemtype"]
    qty = int(row["quantity"])
    
    demand[(o,i)] = demand.get((o,i),0) + qty

# ==================================================
# Item position on conveyor
# ==================================================

x = m.addVars(
    [(item_id,p) for (item_id,i,t) in items for p in positions],
    vtype=GRB.BINARY,
    name="x"
)

m.addConstrs(
    gp.quicksum(x[item_id,p] for p in positions) == 1
    for (item_id,i,t) in items
)

m.addConstrs(
    gp.quicksum(x[item_id,p] for (item_id,i,t) in items) == 1
    for p in positions
)

# ==================================================
# Tote contiguity constraints
# ==================================================

start = m.addVars(totes, vtype=GRB.INTEGER, lb=1, ub=N, name="tote_start")

pos_expr = {
    item_id: gp.quicksum(p * x[item_id,p] for p in positions)
    for (item_id,i,t) in items
}

for (item_id,i,t) in items:

    m.addConstr(pos_expr[item_id] >= start[t])
    m.addConstr(pos_expr[item_id] <= start[t] + tote_size[t] - 1)

# prevent tote overlap

z = m.addVars(totes, totes, vtype=GRB.BINARY, name="tote_order")

M = N

for t1 in totes:
    for t2 in totes:
        if t1 != t2:

            m.addConstr(
                start[t1] + tote_size[t1]
                <= start[t2] + M*(1 - z[t1,t2])
            )

            m.addConstr(
                start[t2] + tote_size[t2]
                <= start[t1] + M*(z[t1,t2])
            )

# ==================================================
# Belt assignment
# ==================================================

belts = range(1,5)

y = m.addVars(
    [(b,o) for b in belts for o in orders],
    vtype=GRB.BINARY,
    name="assign"
)

# Each order must be assigned to exactly one belt.
# A belt can handle multiple orders (or none).
m.addConstrs(
    gp.quicksum(y[b,o] for b in belts) == 1
    for o in orders
)

# ==================================================
# Picking variables
# ==================================================

pick = m.addVars(
    [(b,item_id,o,p) for b in belts
                      for (item_id,i,t) in items
                      for o in orders
                      for p in positions],
    vtype=GRB.BINARY,
    name="pick"
)

m.addConstrs(
    pick[b,item_id,o,p] <= x[item_id,p]
    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

m.addConstrs(
    gp.quicksum(pick[b,item_id,o,p] for b in belts for o in orders) <= 1
    for (item_id,i,t) in items
    for p in positions
)

# belt precedence

m.addConstrs(
    pick[b+1,item_id,o,p] <=
    1 - gp.quicksum(pick[b,item_id,o2,p] for o2 in orders)
    
    for b in range(1,4)
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

# belt must match assigned order

m.addConstrs(
    pick[b,item_id,o,p] <= y[b,o]
    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

# remove invalid picks

for (item_id,i,t) in items:
    for o in orders:
        if (o,i) not in demand:
            for b in belts:
                for p in positions:
                    m.addConstr(pick[b,item_id,o,p] == 0)

# demand satisfaction

for (o,i),qty in demand.items():

    m.addConstr(
        gp.quicksum(
            pick[b,item_id,o,p]
            for b in belts
            for (item_id,it,t) in items if it == i
            for p in positions
        ) == qty
    )

# ==================================================
# Order sequencing on each belt
# (a belt must finish one order before starting another)
# ==================================================

Mpos = N

first_pick = m.addVars(
    [(b,o) for b in belts for o in orders],
    vtype=GRB.INTEGER,
    lb=1,
    ub=N,
    name="first_pick"
)

last_pick = m.addVars(
    [(b,o) for b in belts for o in orders],
    vtype=GRB.INTEGER,
    lb=0,
    ub=N,
    name="last_pick"
)

# Link first/last pick positions to actual picks
for b in belts:
    for o in orders:
        for (item_id,i,t) in items:
            for p in positions:
                m.addConstr(
                    first_pick[b,o] <= p + Mpos * (1 - pick[b,item_id,o,p])
                )
                m.addConstr(
                    last_pick[b,o] >= p - Mpos * (1 - pick[b,item_id,o,p])
                )

# For each belt, enforce a non-overlapping order sequence
order_seq = m.addVars(
    [(b,o1,o2) for b in belts for o1 in orders for o2 in orders],
    vtype=GRB.BINARY,
    name="order_seq"
)

for b in belts:
    for o1 in orders:
        for o2 in orders:
            if o1 == o2:
                continue

            # Only relevant when both orders are actually assigned to this belt
            m.addConstr(
                last_pick[b,o1]
                <= first_pick[b,o2]
                   + Mpos * (3 - order_seq[b,o1,o2] - y[b,o1] - y[b,o2])
            )

            m.addConstr(
                last_pick[b,o2]
                <= first_pick[b,o1]
                   + Mpos * (order_seq[b,o1,o2] + 2 - y[b,o1] - y[b,o2])
            )

# ==================================================
# TIME-BASED OBJECTIVE
# ==================================================

TIME = m.addVar(vtype=GRB.CONTINUOUS, name="completion_time")

m.addConstrs(
    TIME >= (3*(p) + 7.5*(b-1) + 3) *
            pick[b,item_id,o,p]

    for b in belts
    for (item_id,i,t) in items
    for o in orders
    for p in positions
)

m.setObjective(TIME, GRB.MINIMIZE)

# ==================================================
# Solve
# ==================================================

m.optimize()

# ==================================================
# Print Results
# ==================================================

print("\nITEM SEQUENCE ON CONVEYOR")

# Map item types to human-readable names
item_type_names = {
    0: "circle",
    1: "pentagon",
    2: "trapezoid",
    3: "triangle",
    4: "star",
    5: "moon",
    6: "heart",
    7: "cross",
}

sequence = []

for (item_id,i,t) in items:
    for p in positions:
        if x[item_id,p].X > 0.5:
            sequence.append((p,item_id,i,t))

sequence.sort()

for p,item_id,i,t in sequence:
    name = item_type_names.get(i, f"type-{i}")
    print(f"Position {p}: Item {item_id} | Type {i} ({name}) | Tote {t}")

print("\nTOTE SEQUENCE")

for p,item_id,i,t in sequence:
    print(f"Position {p}: Tote {t}")

print("\nBELT → ORDER ASSIGNMENTS")

for b in belts:
    for o in orders:
        if y[b,o].X > 0.5:
            print(f"Belt {b} processes Order {o}")

print("\nPICKS WITH TIME")

# Collect all picks with their times
picks_with_time = []

for b in belts:
    for (item_id,i,t) in items:
        for o in orders:
            for p in positions:
                if pick[b,item_id,o,p].X > 0.5:

                    t_pick = 3*(p) + 7.5*(b-1) + 3

                    picks_with_time.append((t_pick, b, item_id, i, t, o, p))

# Sort by pick time
picks_with_time.sort(key=lambda x: x[0])

# Print in ascending time order
for t_pick, b, item_id, i, t, o, p in picks_with_time:
    print(
        f"Belt {b} picks Item {item_id} "
        f"(Type {i}, Tote {t}) "
        f"for Order {o} at position {p} "
        f"→ Time = {t_pick:.2f} sec"
    )

print("\nSYSTEM COMPLETION TIME:", TIME.X)

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 45295 rows, 7472 columns and 122180 nonzeros
Model fingerprint: 0x8c73ef28
Variable types: 1 continuous, 7471 integer (7418 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e+00, 5e+01]
Presolve removed 40617 rows and 4798 columns
Presolve time: 1.18s
Presolved: 4678 rows, 2674 columns, 19417 nonzeros
Variable types: 0 continuous, 2674 integer (2620 binary)
Found heuristic solution: objective 73.5000000
Found heuristic solution: objective 66.0000000

Root relaxation: objective 1.260000e+01, 2233 iterations, 0.46 seconds (0.20 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | I

# CSV Input File

In [70]:
# Shape mapping
shape_map = {
    0: "circle",
    1: "pentagon",
    2: "trapezoid",
    3: "triangle",
    4: "star",
    5: "moon",
    6: "heart",
    7: "cross"
}

belts = range(1,5)

rows = []

for b in belts:
    row = {"conv_num": b}

    # Initialize columns
    for col in shape_map.values():
        row[col] = 0

    # Count picks by item type
    for item_id, item_type, tote in items:

        if item_type not in shape_map:
            continue

        shape_name = shape_map[item_type]

        count = 0

        for o in orders:
            for p in positions:
                if pick[b, item_id, o, p].X > 0.5:
                    count += 1

        row[shape_name] += count

    rows.append(row)

# Create dataframe
df_picklist = pd.DataFrame(rows)

# Ensure correct column order
df_picklist = df_picklist[
    ["conv_num"] + list(shape_map.values())
]

# Save CSV
output_path = "MIP-belt_picklist.csv"
df_picklist.to_csv(output_path, index=False)

print("CSV generated:", output_path)

CSV generated: MIP-belt_picklist.csv
